# `true_color_with_night_ir` — day/night eclipse frame (pyramids-eo, no satpy)

This notebook renders satpy's `true_color_with_night_ir` **look entirely in pyramids-eo** — no satpy / PyTroll
import. It uses a small **synthetic** scene (no data download) to demonstrate the pipeline end to end:

1. `true_color` — day RGB from calibrated red / blue / near-IR reflectances (CIMSS synthetic green);
2. `night_ir` — RGBA IR cloud image with a cloudiness alpha;
3. `true_color_with_night_ir` — overlay the clouds on a city-lights background and cross-fade against the day
   image by the **solar zenith angle** (`solar_zenith_angle`).

With real EUMETSAT FCI granules the same calls produce the actual eclipse frame; here we use synthetic bands so
the notebook runs offline and deterministically.

In [ ]:
import datetime as dt
import sys

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pyramids_eo.composites import (
    night_ir,
    solar_zenith_angle,
    true_color,
    true_color_with_night_ir,
)

assert "satpy" not in sys.modules, "this pipeline uses no satpy"

In [ ]:
# Synthetic scene on a lon/lat grid spanning a day/night terminator.
rows, cols = 120, 160
lat = np.linspace(70, 30, rows)[:, None] * np.ones((1, cols))
lon = np.linspace(-40, 60, cols)[None, :] * np.ones((rows, 1))

rng = np.random.default_rng(0)
red = 0.30 + 0.10 * rng.random((rows, cols))
blue = 0.40 + 0.10 * rng.random((rows, cols))
nir = 0.50 + 0.10 * rng.random((rows, cols))
day = true_color(red, blue, nir, clip=True)

# Night IR clouds: a cold blob, alpha = cloudiness (city lights show through gaps).
yy, xx = np.mgrid[0:rows, 0:cols]
cloud = np.exp(-(((xx - 100) / 25) ** 2 + ((yy - 40) / 20) ** 2))
clouds = night_ir(0.85 * cloud, 0.75 * cloud, 0.65 * cloud, alpha=cloud)

# Synthetic city-lights background.
background = np.full((3, rows, cols), 0.04)
lights = rng.random((rows, cols)) > 0.985
background[0][lights], background[1][lights], background[2][lights] = 1.0, 0.9, 0.6

# Solar zenith angle from the Sun's real geometry (fixed eclipse-day time).
when = dt.datetime(2026, 8, 12, 18, 0, tzinfo=dt.timezone.utc)
sza = solar_zenith_angle(when, lat=lat, lon=lon)
print("SZA range (deg):", round(float(sza.min()), 1), "->", round(float(sza.max()), 1))

In [ ]:
frame = true_color_with_night_ir(day, clouds, background, sza)
rgb = np.clip(np.moveaxis(frame, 0, -1), 0.0, 1.0)  # (H, W, 3) for imshow

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(rgb)
ax.set_title("Synthetic true_color_with_night_ir (day/night blend by SZA)")
ax.axis("off")
print("frame shape:", frame.shape)

The sunlit side renders as `true_color`; the night side shows the city-lights background with the IR clouds
overlaid, cross-faded smoothly through the terminator by the solar zenith angle — the `true_color_with_night_ir`
look, with no satpy dependency.